Starting with $u(x,y)=21-4x+3y$.  
Boundary conditions are $u=1$ on left and right sides, and $\frac{\partial u}{\partial n}=0$ on top and bottom  
Using the PDE he gave us,   
$-\nabla \cdot(K(x,y) \nabla u) + \sigma^2 u = f$  strong form  
aka $-\left[\frac{\partial}{\partial x}(K(x,y) \frac{\partial}{\partial x}u) + \frac{\partial}{\partial y}(K(x,y) \frac{\partial}{\partial y} u)\right] + \sigma^2 u = f$  
$\int_\Omega\left[\left(K(x,y) \frac{\partial}{\partial x} u \frac{\partial}{\partial x} \phi + K(x,y) \frac{\partial}{\partial y} u \frac{\partial}{\partial y} \phi\right) + \sigma^2 u \phi \right] dxdy= \int_\Omega f \phi dxdy$   weak form  
where $K(x,y)=1+x^2+y^2$ and $\sigma=\frac{1}{2}$.
    
Results: Does not seem to match completely

<span style="color:red">Current Concern:  Why doesn't it match?  
Consider error calculations in addition to visual inspection.</span>


In [ ]:
# Import any packages needed
from ngsolve import *
from ngsolve.webgui import Draw
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# Domain is unit square, create initial mesh
N = 64 
mesh = Mesh(unit_square.GenerateMesh(maxh=1/N))
mesh.nv, mesh.ne
Draw(mesh);

In [ ]:
# Working with H1-conforming piecewise linear finite elements
# boundary conditions are Dirichlet on the left and right and default=Neumann on the other 2 sides
fes = H1(mesh, order=1, dirichlet="left|right")
fes.ndof

# Set test and trial spaces
u = fes.TrialFunction()
phi = fes.TestFunction()

# Coefficient function for K(x,y)
Kxy = CoefficientFunction(1+x*x+y*y)
# Kxy = CoefficientFunction(3)
sigma = 0.5

# Bilinear form
a = BilinearForm(fes)
a += Kxy*grad(u)*grad(phi)*dx+sigma*sigma*u*phi*dx
a.Assemble()

print(a.mat)

In [ ]:
# Generating boundary values for Dirichlet boundaries and RHS from u=21-4x+3y 
# then that's what we are hoping to see upon solve
plant = CoefficientFunction(21 - 4 * x + 3 * y)

# Right hand side
f = LinearForm(fes)
# generated f from  f=-div(k*grad(plant))+sigma^2*plant
fant = 7 * x - 21/4 * y + 21/4
f += fant*phi*dx
f.Assemble()

print(f.vec)

In [ ]:
# Solution will be stored as a grid function
gfu = GridFunction(fes)
gfu.Set(plant, BND) # this sets the Dirichlet boundary component to 1, might want to modify later
Draw(gfu) # at this point only the BCs are in effect

In [ ]:
# To see the K(x,y) function, we plot it on our mesh
Draw(Kxy, mesh)

In [ ]:
# Solve the PDE using CG
c = Preconditioner(a,"local")
c.Update()
solvers.BVP(bf=a, lf=f, gf=gfu, pre=c, maxsteps=500, tol=1e-14)
Draw(gfu)

In [ ]:
Draw(plant,mesh) # original function for visual comparison